# Hand tracking and demo extraction

How a phone clip becomes a table-frame `HumanDemo`: MediaPipe hand landmarks + `solvePnP` for a metric TCP, ArUco for the table/bowl/plate frames, then smoothing, 20 Hz resampling, grasp/release segmentation and a depth anchor.

- `ego2robo.hand`: `HandTcpTracker`
- `ego2robo.markers`: `MarkerPoseEstimator`
- `ego2robo.extract`: `DemoExtractor`, `ClipReport` (the QC gates)

**Stand-in data.** No real capture has been processed yet. The hand is a real hand from a third-party GIF (`data/raw/overview_gif/first_clip/`, not committed, not part of the submission dataset); intrinsics are guessed (fx = fy = 800 px); the ArUco markers are *synthetic*, pasted in at known poses so the recovered poses can be checked against ground truth. The cube in the GIF is not where the synthetic bowl marker is, so the QC gates are expected to flag this clip.

In [1]:
from ego2robo.utils.notebook import setup_notebook

ROOT = setup_notebook()  # chdir to the repo root and set MUJOCO_GL=egl; must precede any mujoco import

In [2]:
import tempfile
from pathlib import Path

from ego2robo.hand import track_video
from ego2robo.utils.dashboard_video import write_tracking_dashboard_video
from ego2robo.utils.notebook import show_video
from ego2robo.utils.synthetic import StandInScene
from ego2robo.utils.transforms import pose_error
from ego2robo.utils.video import write_tcp_overlay_video
from ego2robo.utils.viewer import aperture_figure, tracking_figure, trajectory_figure

# Real hand frames (GIF frames 20-45, 10 fps) with ArUco markers pasted in at known poses.
SCENE = StandInScene.from_clip_dir(ROOT / "data/raw/overview_gif/first_clip")
INTRINSICS = SCENE.intrinsics
WORK_DIR = Path(tempfile.mkdtemp())
CLIP = SCENE.render_clip(WORK_DIR / "clip.mp4")
print(len(SCENE.frame_paths), "frames ->", CLIP.name)

26 frames -> clip.mp4


## Per-frame hand tracking

`HandTcpTracker` returns the TCP (thumb/index-tip midpoint) in the camera frame, the pinch aperture, and the PnP reprojection error. The reprojection error is the per-frame confidence signal: a PnP solution that diverges shows up here as thousands of px (see the note below the plot).

In [3]:
tracked = track_video(CLIP, INTRINSICS)
print(f"hand detected in {sum(f.result is not None for f in tracked)}/{len(tracked)} frames")
tracking_figure(tracked).show()

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1791680172.023071  403314 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1791680172.033296  403314 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1791680172.068876  403315 landmark_projection_calculator.cc:81] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


hand detected in 15/26 frames


In [4]:
overlay = write_tcp_overlay_video(tracked, INTRINSICS, WORK_DIR / "tcp_overlay.mp4", fps=SCENE.fps)
show_video(overlay)

### Synced dashboard

The same clip with every view on one clock: camera overlay, the TCP path in the camera frame (grey = whole path, blue = so far, red dot = now) and the pinch aperture with a time cursor. Frames without a hand drop the red dots and show a gap in the curve.

In [5]:
dashboard = write_tracking_dashboard_video(
    tracked,
    INTRINSICS,
    WORK_DIR / "dashboard.mp4",
    fps=SCENE.fps,
    close_threshold_m=0.03,
    open_threshold_m=0.05,
)
show_video(dashboard, width=960)

The first three detected frames are the regression case: `cv2.solvePnP` with `SOLVEPNP_ITERATIVE` from its default guess diverged there (z ~ 0 m, reprojection error up to 7,000 px). The tracker now seeds with EPnP, refines with LM, and rejects solutions with points behind the camera (`tests/test_hand_tracker.py`).

## Whole clip -> `HumanDemo`

`DemoExtractor` ties the tracker and the marker estimator together. With a static camera, `T_cam_table` is one average over the clip (less jitter than per-frame), the bowl/plate poses come from the first 0.6 s before the hand arrives, and the TCP trajectory is gap-filled, smoothed, resampled to 20 Hz, segmented, and depth-anchored at the grasp.

In [6]:
demo, report = SCENE.extract(CLIP)

print(f"usable: {report.usable}")
for reason in report.reasons:
    print("  -", reason)
print(
    f"table marker seen in {report.table_marker_fraction:.0%} of frames; "
    f"anchor scale {report.anchor_scale:.2f}; {len(demo.timestamps_s)} samples at 20 Hz"
)

W0000 00:00:1791680174.036748  403504 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1791680174.046118  403504 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


usable: False
  - no release detected
  - grasp 0.14 m from bowl marker
table marker seen in 100% of frames; anchor scale 1.13; 33 samples at 20 Hz


### Marker poses against ground truth

The synthetic markers have known poses, so this is a real check of the ArUco -> table-frame chain (including the mp4 compression).

In [7]:
for name, estimated, truth in (
    ("T_table_cam", demo.t_table_cam, SCENE.t_table_cam),
    ("T_table_bowl", demo.t_table_bowl, SCENE.t_table_bowl),
    ("T_table_plate", demo.t_table_plate, SCENE.t_table_plate),
):
    mm, deg = pose_error(estimated, truth)
    print(f"{name:14s} {mm:5.1f} mm  {deg:4.2f} deg")

T_table_cam      8.0 mm  0.47 deg
T_table_bowl     2.0 mm  0.41 deg
T_table_plate    1.6 mm  0.48 deg


In [8]:
trajectory_figure(demo.tcp_pos_table, demo.is_grasping, "TCP, table frame").show()
aperture_figure(
    demo.timestamps_s,
    demo.pinch_aperture_m,
    close_threshold_m=0.03,
    open_threshold_m=0.05,
    grasp_frame_idx=demo.grasp_frame_idx,
    release_frame_idx=demo.release_frame_idx,
).show()

## Reading the result

- **Marker poses** recover to a few mm / under a degree. The camera pose carries the largest error (6 cm marker, 0.6 m away); the real table marker should be larger (>= 10 cm).
- **QC gates** flag this clip, correctly: the TCP is nowhere near the synthetic bowl marker, and the hand leaves the frame still pinching, so there is no release. Flagged clips are reported and excluded from export unless asked for, never silently dropped.
- **Fewer detections than the raw clip** (14/26 here, 17 of 18 hand frames on the unmodified frames): the synthetic markers are pasted *on top of* the hand pixels, so they partly hide it. Real markers lie on the table and the hand occludes them, which is the opposite failure (a marker lost, not a hand lost).
- **Not validated here:** metric depth. The intrinsics are a guess and there is no ground-truth hand position, so `z_cam` is only plausible, not verified. That needs a real clip with the hand placed on a marker.
- The anchor correction is a single scale per clip about the camera origin (it fixes depth, not lateral error), and the aperture is left uncorrected.